# Ensemble methods

This script demonstrates ensemble learning by combining three different models—Decision Tree, Logistic Regression, and a Neural Network—to classify handwritten digits from the MNIST dataset.

This setup prepares MNIST for an ensemble experiment mixing classic machine-learning models with a neural network. After importing TensorFlow/Keras and scikit-learn utilities, the MNIST digits dataset is loaded and normalised by scaling pixel values from 0–255 to 0–1 for stable optimisation. Because models like decision trees and logistic regression expect vector inputs rather than images, the 28×28 images are flattened into 784-dimensional feature vectors (x_train_flat, x_test_flat). In parallel, labels are one-hot encoded (y_train_oh, y_test_oh) to suit the neural network’s softmax output over 10 digit classes. This dual preparation lets us train both non-CNN classifiers on flattened data and a Keras model on image labels in a single workflow, enabling later combination in an ensemble.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten for non-CNN models
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

# One-hot encode labels for the neural network
y_train_oh = to_categorical(y_train, 10)
y_test_oh = to_categorical(y_test, 10)

This section trains three base classifiers that we’ll later combine in an ensemble. First, a Decision Tree with max_depth=10 (to limit overfitting) is fitted on the 784-dimensional flattened images. Next, Logistic Regression is trained on the same features with max_iter=100 to ensure convergence, providing a strong linear baseline. Finally, a simple neural network is built as a multilayer perceptron: two dense layers (128 ReLU units, then 64 ReLU units) feed into a 10-way softmax output. The network is compiled with Adam and categorical cross-entropy and trained for 3 epochs with batches of 128 on the one-hot labels. Together, these models capture complementary inductive biases—nonlinear splits (tree), linear separability (logistic), and learned feature transformations (NN)—setting the stage for an ensemble that can outperform any single model.

In [ ]:
# Train the individual models

# (a) Decision Tree
print("Training Decision Tree...")
tree_clf = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_clf.fit(x_train_flat, y_train)

# (b) Logistic Regression
print("Training Logistic Regression...")
log_reg = LogisticRegression(max_iter=100, random_state=42)
log_reg.fit(x_train_flat, y_train)

# (c) Neural Network
print("Training Neural Network...")
nn_model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
nn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
nn_model.fit(x_train_flat, y_train_oh, epochs=3, batch_size=128, verbose=1)

This code defines a simple ensemble model that combines different classifiers through majority voting. The EnsembleModel class takes a list of trained models as input. When making predictions, it loops through each model: for scikit-learn models (decision tree and logistic regression), it directly calls .predict, while for the Keras neural network it uses .predict followed by argmax to obtain class labels. These predictions are collected, stacked into a single array, and passed to a voting step where the most common class label across models is selected for each input sample. By combining diverse models—each with different decision strategies—the ensemble can reduce individual weaknesses and produce more reliable predictions. In this case, the ensemble integrates the decision tree, logistic regression, and neural network into one unified classifier.

In [ ]:
# Create the ensemble model
class EnsembleModel:
    def __init__(self, models):
        """
        Ensemble of models combining predictions via majority voting.
        :param models: List of models to include in the ensemble.
        """
        self.models = models

    def predict(self, x):
        """
        Combines predictions from all models using majority voting.
        :param x: Input data.
        :return: Final predictions (class labels).
        """
        # Collect predictions from all models
        predictions = []
        for model in self.models:
            if isinstance(model, models.Sequential):  # Neural network
                preds = np.argmax(model.predict(x), axis=1)
            else:  # Scikit-learn models
                preds = model.predict(x)
            predictions.append(preds)

        # Stack predictions and compute majority vote
        predictions = np.array(predictions)
        majority_vote = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions)
        return majority_vote

# Combine all models in an ensemble
ensemble = EnsembleModel([tree_clf, log_reg, nn_model])

This function evaluates the performance of any given model by comparing its predictions to the true test labels. It calls the model’s predict method on the test inputs to generate class predictions, then calculates accuracy using scikit-learn’s accuracy_score, which measures the proportion of correctly classified samples. The result is printed as a percentage with two decimal places for readability and also returned for further use. When applied to the ensemble, this evaluation allows us to directly compare its accuracy with the individual models trained earlier, highlighting whether combining classifiers improves overall performance.

In [ ]:
# Evaluate the ensemble
def evaluate_model(model, x_test, y_test):
    """
    Evaluates the model's accuracy.
    :param model: Model to evaluate.
    :param x_test: Test inputs.
    :param y_test: True labels.
    :return: Accuracy.
    """
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    return accuracy

This final block compares the performance of the individual classifiers against the ensemble. The decision tree and logistic regression models are evaluated directly using the evaluate_model function on the flattened test set. The neural network requires an additional step: predictions are generated with .predict, then converted into class labels with argmax before computing accuracy with scikit-learn. After printing the accuracy of each standalone model, the ensemble is evaluated in the same way, showing whether majority voting across diverse classifiers provides a measurable boost in overall performance. This side-by-side comparison demonstrates the central motivation of ensemble methods: leveraging complementary strengths to outperform individual models.

In [ ]:
print("Evaluating individual models...")
tree_acc = evaluate_model(tree_clf, x_test_flat, y_test)
log_reg_acc = evaluate_model(log_reg, x_test_flat, y_test)

nn_preds = np.argmax(nn_model.predict(x_test_flat), axis=1)
nn_acc = accuracy_score(y_test, nn_preds)
print(f"Neural Network Accuracy: {nn_acc * 100:.2f}%")

print("\nEvaluating ensemble model...")
ensemble_acc = evaluate_model(ensemble, x_test_flat, y_test)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training Decision Tree...
Training Logistic Regression...


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Neural Network...
Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8260 - loss: 0.6113
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9580 - loss: 0.1449
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9713 - loss: 0.0954
Evaluating individual models...
Accuracy: 86.63%
Accuracy: 92.56%
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 981us/step
Neural Network Accuracy: 97.03%

Evaluating ensemble model...
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 911us/step
Accuracy: 95.25%
